<a href="https://colab.research.google.com/github/ahmad0719/cal/blob/main/Copy_of_AhmadAssad_CIFAR10_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN Image Classification — CIFAR-10
### Tasks 1–4: CNN, Architecture Modification, Data Augmentation, and Transfer Learning

This Google Colab notebook implements the requested deep-learning experiments on CIFAR-10.

**Tasks covered**
1. Build and evaluate a CNN with at least 3 convolutional layers and 2 max-pooling layers.
2. Modify the CNN architecture and compare performance.
3. Apply rotation, zoom, horizontal flip, and brightness augmentation and compare performance.
4. Use transfer learning with a pretrained CNN (MobileNetV2) adapted to CIFAR-10.

> **Note:** Transfer learning uses ImageNet-pretrained MobileNetV2. CIFAR-10 images are resized to 96×96 for this experiment.


In [ ]:
# Install/import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


TensorFlow version: 2.20.0
GPU available: []


## 1. Load and preprocess CIFAR-10

In [ ]:
# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Training images:", x_train.shape)
print("Test images:", x_test.shape)

# Normalize pixel values from [0, 255] to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flatten labels from shape (n, 1) to (n,)
y_train = y_train.squeeze()
y_test = y_test.squeeze()

print("Normalized range:", x_train.min(), "to", x_train.max())

 20217856/170498071 ━━━━━━━━━━━━━━━━━━━━ 25:44 10us/step

In [ ]:
# Visualize sample training images
plt.figure(figsize=(10, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()


## Task 1 — Build a Simple CNN

Architecture:
- Conv2D: 32 filters
- MaxPooling
- Conv2D: 64 filters
- MaxPooling
- Conv2D: 128 filters
- Flatten
- Dense + Dropout
- 10-class softmax output

The model uses **Adam** and **sparse categorical cross-entropy**, which is appropriate for integer class labels.


In [ ]:
def build_original_cnn():
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

original_model = build_original_cnn()
original_model.summary()


In [ ]:
# Train Task 1 model
EPOCHS = 10
BATCH_SIZE = 64

history_original = original_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

original_test_loss, original_test_acc = original_model.evaluate(
    x_test, y_test, verbose=0
)

print(f"Original CNN test loss: {original_test_loss:.4f}")
print(f"Original CNN test accuracy: {original_test_acc:.4f} ({original_test_acc*100:.2f}%)")


In [ ]:
def plot_history(history, title):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="Training")
    plt.plot(history.history["val_accuracy"], label="Validation")
    plt.title(title + " — Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="Training")
    plt.plot(history.history["val_loss"], label="Validation")
    plt.title(title + " — Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_history(history_original, "Original CNN")


## Task 1 — Test Images and Predicted Labels

In [ ]:
# Visualize predictions
pred_probs = original_model.predict(x_test[:12], verbose=0)
pred_labels = np.argmax(pred_probs, axis=1)

plt.figure(figsize=(12, 8))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[i])
    true_name = class_names[y_test[i]]
    pred_name = class_names[pred_labels[i]]
    title = f"True: {true_name}\nPred: {pred_name}"
    plt.title(title, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()


## Task 2 — Modify the CNN Architecture

The modified model:
- Uses **LeakyReLU** instead of standard ReLU activations.
- Increases the convolutional filters to 64 → 128 → 256.
- Adds Batch Normalization.
- Keeps two max-pooling layers.
- Uses Global Average Pooling instead of Flatten to reduce the number of parameters.

This provides a meaningful architectural experiment rather than merely changing one hyperparameter.


In [ ]:
def build_modified_cnn():
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Conv2D(64, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.1)(x)
    x = layers.Conv2D(64, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.1)(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.1)(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(256, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.1)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128)(x)
    x = layers.LeakyReLU(negative_slope=0.1)(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

modified_model = build_modified_cnn()
modified_model.summary()


In [ ]:
history_modified = modified_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

modified_test_loss, modified_test_acc = modified_model.evaluate(
    x_test, y_test, verbose=0
)

print(f"Modified CNN test loss: {modified_test_loss:.4f}")
print(f"Modified CNN test accuracy: {modified_test_acc:.4f} ({modified_test_acc*100:.2f}%)")

plot_history(history_modified, "Modified CNN")


In [ ]:
# Task 2 comparison
comparison = pd.DataFrame({
    "Model": ["Original CNN", "Modified CNN"],
    "Test Loss": [original_test_loss, modified_test_loss],
    "Test Accuracy": [original_test_acc, modified_test_acc]
})

comparison


### Task 2 — Findings

The exact numerical result depends on the Colab runtime, random initialization, and number of epochs. In general:

- **More filters** increase representational capacity but also computational cost.
- **LeakyReLU** allows a small gradient for negative inputs and can reduce inactive-neuron behavior.
- **Batch Normalization** can stabilize and accelerate training.
- **Global Average Pooling** substantially reduces parameters compared with Flatten + a large dense layer.
- A more complex architecture is not automatically better; validation/test performance and training behavior determine whether the modification helped.


## Task 3 — Data Augmentation

In [ ]:
# Data augmentation:
# rotation, zoom, horizontal flip, and brightness adjustment
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.15)
], name="data_augmentation")

# Visualize augmented versions of one image
sample = tf.expand_dims(x_train[0], 0)

plt.figure(figsize=(10, 6))
for i in range(8):
    augmented = data_augmentation(sample, training=True)[0]
    plt.subplot(2, 4, i + 1)
    plt.imshow(tf.clip_by_value(augmented, 0, 1))
    plt.axis("off")
plt.suptitle("Examples of Augmented CIFAR-10 Images")
plt.tight_layout()
plt.show()


In [ ]:
def build_augmented_cnn():
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        data_augmentation,

        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, activation="relu", padding="same"),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

augmented_model = build_augmented_cnn()
augmented_model.summary()


In [ ]:
history_augmented = augmented_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

augmented_test_loss, augmented_test_acc = augmented_model.evaluate(
    x_test, y_test, verbose=0
)

print(f"Augmented CNN test loss: {augmented_test_loss:.4f}")
print(f"Augmented CNN test accuracy: {augmented_test_acc:.4f} ({augmented_test_acc*100:.2f}%)")

plot_history(history_augmented, "Augmented CNN")

In [ ]:
# Task 3 comparison
comparison = pd.DataFrame({
    "Model": ["Original CNN", "Augmented CNN"],
    "Test Loss": [original_test_loss, augmented_test_loss],
    "Test Accuracy": [original_test_acc, augmented_test_acc]
})

comparison


### Task 3 — Findings

Data augmentation generates varied training examples without changing the class labels. Horizontal flipping, small rotations, zooming, and brightness changes can improve generalization by reducing dependence on exact training-image appearances.

The augmented model may show **lower training accuracy** while still achieving comparable or better validation/test accuracy. That pattern can indicate reduced overfitting.

> Augmentation is applied only during training. The test set remains unchanged so that evaluation remains fair.


## Task 4 — Transfer Learning

We use **MobileNetV2 pretrained on ImageNet** as the feature extractor.

Because CIFAR-10 images are 32×32, they are resized to 96×96 before being passed to MobileNetV2. The ImageNet classification head is removed and replaced with a CIFAR-10 classification head.

The pretrained base is initially frozen. This is followed by a short optional fine-tuning stage.


In [ ]:
# Resize CIFAR-10 images for MobileNetV2.
# 96x96 is used to keep the experiment practical in Colab.
TRANSFER_SIZE = 96

x_train_transfer = tf.image.resize(x_train, (TRANSFER_SIZE, TRANSFER_SIZE))
x_test_transfer = tf.image.resize(x_test, (TRANSFER_SIZE, TRANSFER_SIZE))

print(x_train_transfer.shape, x_test_transfer.shape)


In [ ]:
# Build transfer-learning model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(TRANSFER_SIZE, TRANSFER_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

transfer_model = keras.Sequential([
    layers.Input(shape=(TRANSFER_SIZE, TRANSFER_SIZE, 3)),
    # MobileNetV2 expects inputs in the [-1, 1] range.
    layers.Rescaling(scale=2.0, offset=-1.0),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

transfer_model.summary()


In [ ]:
# Train the new classification head while MobileNetV2 is frozen
TRANSFER_EPOCHS = 5

history_transfer = transfer_model.fit(
    x_train_transfer, y_train,
    validation_split=0.1,
    epochs=TRANSFER_EPOCHS,
    batch_size=64,
    verbose=1
)

transfer_test_loss, transfer_test_acc = transfer_model.evaluate(
    x_test_transfer, y_test, verbose=0
)

print(f"Transfer learning test loss: {transfer_test_loss:.4f}")
print(f"Transfer learning test accuracy: {transfer_test_acc:.4f} ({transfer_test_acc*100:.2f}%)")

plot_history(history_transfer, "Transfer Learning")


### Optional Fine-Tuning

After training the new classifier head, a small number of the later MobileNetV2 layers can be unfrozen. Fine-tuning should use a **very small learning rate** so that useful pretrained features are not destroyed.

This stage is optional because it increases training time considerably.


In [ ]:
# Optional fine-tuning of the last portion of MobileNetV2
base_model.trainable = True

# Freeze most layers and fine-tune only the final ~20 layers.
for layer in base_model.layers[:-20]:
    layer.trainable = False

transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

FINE_TUNE_EPOCHS = 3

history_finetune = transfer_model.fit(
    x_train_transfer, y_train,
    validation_split=0.1,
    epochs=FINE_TUNE_EPOCHS,
    batch_size=64,
    verbose=1
)

finetune_test_loss, finetune_test_acc = transfer_model.evaluate(
    x_test_transfer, y_test, verbose=0
)

print(f"Fine-tuned test loss: {finetune_test_loss:.4f}")
print(f"Fine-tuned test accuracy: {finetune_test_acc:.4f} ({finetune_test_acc*100:.2f}%)")


In [ ]:
# Final comparison across all experiments
final_comparison = pd.DataFrame({
    "Model": [
        "Original CNN",
        "Modified CNN",
        "CNN + Data Augmentation",
        "MobileNetV2 Transfer Learning",
        "MobileNetV2 Fine-Tuned"
    ],
    "Test Loss": [
        original_test_loss,
        modified_test_loss,
        augmented_test_loss,
        transfer_test_loss,
        finetune_test_loss
    ],
    "Test Accuracy": [
        original_test_acc,
        modified_test_acc,
        augmented_test_acc,
        transfer_test_acc,
        finetune_test_acc
    ]
})

final_comparison["Test Accuracy (%)"] = final_comparison["Test Accuracy"] * 100
final_comparison


In [ ]:
# Plot final test accuracy comparison
plt.figure(figsize=(10, 5))
plt.bar(final_comparison["Model"], final_comparison["Test Accuracy (%)"])
plt.ylabel("Test Accuracy (%)")
plt.xlabel("Model")
plt.title("CIFAR-10 Model Performance Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


# Overall Conclusions

1. **Task 1:** The baseline CNN learns hierarchical visual features through multiple convolutional layers and pooling operations.
2. **Task 2:** Increasing model capacity and using LeakyReLU/Batch Normalization changes optimization and representational power; the test results determine whether the modification is beneficial.
3. **Task 3:** Data augmentation increases training diversity and can improve generalization, especially when the baseline model overfits.
4. **Task 4:** Transfer learning starts from features learned on a large external dataset. A pretrained network can provide strong representations even when the target dataset is relatively small.
5. The **final comparison table and plot** provide the empirical basis for selecting the best-performing approach.

### Reproducibility
Results can vary slightly between runs because of random initialization and training nondeterminism. For a fair comparison, the models above use fixed dataset splits and comparable training settings where practical.
